# Drift polynomial distillation
Only constants, powers and monomial interactions. Observed states and teacher-simulated query states are distinguished. Transforms are fit on symbolic training states only; clone groups do not overlap symbolic train/validation/test. Teacher/upstream-PCA exposure remains. Validation selects hyperparameters; test is evaluated after freezing selection.

In [ ]:
from pathlib import Path
import os,sys,json
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
import pandas as pd,numpy as np
from IPython.display import display,Image,Markdown


In [ ]:
case50=json.loads(Path('outputs/sindy/drift/results.json').read_text())
display(pd.read_csv('outputs/sindy/drift/sweep.csv'))
print('50-PC function success:',case50['function_success'])
print('50-PC validation NRMSE:',case50['validation']['nrmse'])
print('50-PC all-domain validation:',case50['validation_all_domain']['nrmse'])
display(Image(filename='outputs/figures/drift_sparsity.png'))

The 50-PC trial is insufficient, not a symbolic-SDE success. The training-defined norm domain is a local proof-of-concept assumption, with all-domain errors and coverage also reported. Its best sparse selection has material error and extrapolation sensitivity. Degree3 at 50 PCs was not attempted because it would have 23426 features before output expansion.

In [ ]:
root=Path('outputs/sindy_pc5/drift')
assert (root/'results.json').exists(), 'Five-PC experiment must finish first'
case5=json.loads((root/'results.json').read_text())
display(pd.read_csv(root/'sweep.csv'))
display(pd.DataFrame([{ 'partition':name, **case5[name]} for name in ('train','validation','test','test_in_domain')]))
print('Selected:',case5['selected'])
print('Bootstrap support frequency:',case5['mean_selected_frequency'])
print('Threshold +/-10%:',case5['threshold_jitter'])
print('Success gate:',case5['function_success'])
display(Image(filename='outputs/figures/pc5_drift_sparsity.png'))

In [ ]:
from src.distill import PolynomialField
field=PolynomialField.load(root/'model.npz')
display(Markdown((root/'equations.md').read_text()))
print('Shape:',field.predict(np.zeros((3,5))).shape)

NRMSE denominator is RMS teacher target, not its standard deviation. Thresholds use training-RMS-scaled library/targets and retained ridge. Bootstrap holds the coordinate transform fixed and resamples whole clone groups; only five replicates are used for this demo. PC formulas are teacher approximations, not gene-level mechanisms or ground-truth SDE discovery.